<a href="https://colab.research.google.com/github/sms0832/Geol11_F24/blob/main/Geol11_F26_Intro_to_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Python for Geoscientists (1-Hour Crash Course)

Welcome! You already know how to organize data, calculate statistics, and build charts in **Excel and Google Sheets**. Today, we are going to redo several of those exact same analyses in **Python**—using the same Sacramento temperature and climate datasets you worked with earlier this semester.

### Why bother learning Python as a geology major?
* **Reproducibility:** Your code is a written recipe. If you collect new field data next month, you just hit *Run* instead of re-clicking through 20 chart menus.
* **Big datasets:** Spreadsheets slow down or freeze on seismic, LiDAR, or multi-year sensor files. Python handles millions of rows in seconds.
* **Publication-ready figures:** By the end of this hour, you will have a **copy-paste plotting template** (Section 6) you can save and reuse in Geophysics, Hydrogeology, Petrology, Structural Geology, and your Senior Thesis.

### How Google Colab Works (30-Second Survival Guide)
1. **Run a cell:** Click inside a gray code cell and press **`Shift + Enter`** (or click the **▶** play button on the left).
2. **Order matters:** Run cells **from top to bottom**. Python remembers variables you create in earlier cells.
3. **Comments (`#`):** Anything after a `#` symbol is a note for humans—Python completely ignores it.

### Excel / Google Sheets → Python Translator
| What you do in Excel / Sheets | How you do it in Python |
| :--- | :--- |
| Open a spreadsheet tab | Load a **`DataFrame`** (usually named `df`) |
| Write `=AVERAGE()`, `=MIN()`, `=MAX()`, `=STDEV()` | `df.describe()` (one line!) |
| Drag a formula down 2,000 rows | `df['NewCol'] = df['OldCol'] * 2` (instant!) |
| Match two tables with `VLOOKUP` / copy-paste | `pd.merge(table1, table2, on='Year')` |
| Click *Insert → Chart* + *Add Trendline* | `matplotlib` + `linregress` |

---
## Part 1: Load Your Toolbox & Read Data

Python starts out bare-bones, like an empty workbench. Before we do anything, we **import** four free scientific libraries ("toolboxes") that geoscientists use every day:
* **`numpy` (`np`)** — fast math, arrays, and constants like $\pi$
* **`pandas` (`pd`)** — reads CSV and Excel files into spreadsheet-like tables called **DataFrames**
* **`matplotlib.pyplot` (`plt`)** — builds customizable, publication-quality plots
* **`scipy.stats`** — quick statistical tools like linear regression ($R^2$, slope, p-value)

Run the cell below to load them. You won't see any output—no error message means it worked!

In [ ]:
# Run this cell first (Shift + Enter) — no need to edit it
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

### Step 1.1: Load a CSV directly from the web
Instead of downloading a CSV and uploading it to Colab, `pd.read_csv()` can read a file straight from a GitHub URL.

Notice how we refer to a single column using **square brackets and quotes**: `df['Date']`.
> **Pro tip:** Always use bracket notation (`df['Column Name']`) instead of dot notation (`df.Column`). Brackets work *every time*, even when column names have spaces or units like `Mass (GT)`.

In [ ]:
# 1. Store the web address of our Sacramento temperature CSV
url = "https://raw.githubusercontent.com/sms0832/Geol11_F24/refs/heads/main/SacTemps.csv"

# 2. Read the CSV into a pandas DataFrame (think of 'df' as a spreadsheet tab)
df = pd.read_csv(url)

# 3. Tell Python that the 'Date' column contains calendar dates, not plain text
df['Date'] = pd.to_datetime(df['Date'], format='mixed')

# 4. Preview the first 5 rows (just to make sure it loaded cleanly)
df.head()

### Step 1.2: Instant summary statistics
In Excel, getting the count, mean, standard deviation, min, quartiles, and max requires 7 separate formulas per column. In Python, it's **one line**:

In [ ]:
# Quick summary statistics for every numeric column
df.describe()

---
## Part 2: Quick Exploratory Plots vs. Clean Figures

Geoscientists make two kinds of plots:
1. **Quick exploratory plots** (2–3 lines) to check your data for outliers, gaps, or weird spikes.
2. **Clean, labeled figures** (8–10 lines) ready to drop into a lab report, poster, or thesis.

Let's start with two quick exploratory plots—a **histogram** and a **boxplot**—then build a clean time-series figure.

In [ ]:
# Quick histogram of daily Maximum temperature
df['Max'].hist(bins=20, edgecolor='white')
plt.title('Distribution of Daily Max Temperatures in Sacramento')
plt.xlabel('Temperature (°F)')
plt.ylabel('Number of Days')
plt.show()

In [ ]:
# Quick side-by-side boxplot of Min and Max temperatures
# Using pandas .boxplot() automatically labels columns and safely ignores any missing (NaN) values
df[['Min', 'Max']].boxplot()
plt.title('Sacramento Daily Temperature Range')
plt.ylabel('Temperature (°F)')
plt.show()

### Step 2.1: Building a clean time-series plot
Now let's plot both `Max` and `Min` temperatures over time using `matplotlib`. Notice the pattern—**almost every scientific plot in Python follows these same 5 steps**:
1. Create the figure canvas (`plt.figure(...)`)
2. Plot your data (`plt.plot(...)` or `plt.scatter(...)`)
3. Add title and axis labels with units (`plt.title`, `plt.xlabel`, `plt.ylabel`)
4. Add finishing touches (`plt.legend()`, `plt.grid()`)
5. Display it (`plt.show()`)

In [ ]:
# 1. Create a crisp figure canvas (width=9 inches, height=4 inches, 150 dots-per-inch)
plt.figure(figsize=(9, 4), dpi=150)

# 2. Plot Max (red upward triangles, 'r^') and Min (blue downward triangles, 'bv')
#    markerfacecolor='none' makes hollow symbols; alpha=0.6 adds slight transparency
plt.plot(df['Date'], df['Max'], 'r^', markerfacecolor='none', alpha=0.6, label='Daily Max')
plt.plot(df['Date'], df['Min'], 'bv', markerfacecolor='none', alpha=0.6, label='Daily Min')

# 3. Add a clear title and axis labels (always include units!)
plt.title('Maximum and Minimum Daily Temperature in Sacramento')
plt.xlabel('Year')
plt.ylabel('Temperature (°F)')

# 4. Add a legend and subtle gridlines
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)

# 5. Adjust margins cleanly and display the plot
plt.tight_layout()
plt.show()

---
## Part 3: Doing Math on Whole Columns (No Dragging Formulas!)

In Excel, converting a column from Fahrenheit to Celsius means typing a formula and dragging it down thousands of rows. In Python, you write the formula once and it applies to the **entire column** instantly.

Recall the conversion formula:
$$T_{\text{C}} = (T_{\text{F}} - 32) \times \frac{5}{9}$$

> **Watch your parentheses!** Just like in Excel, Python follows PEMDAS (order of operations). Leaving out the parentheses around `(df['Max'] - 32)` would multiply `32 * (5/9)` first and give the wrong temperature!

In [ ]:
# Create a brand-new column 'Max_C' converted from Fahrenheit to Celsius
df['Max_C'] = (df['Max'] - 32) * (5 / 9)

# Check the first 5 rows to verify the numbers make sense!
df.head()

### Your Turn #1: Convert Minimum Temperature to Celsius
In the empty cell below, create a new column called `'Min_C'` that converts `df['Min']` from Fahrenheit to Celsius, then display the first 5 rows with `df.head()`.

In [ ]:
# 1. Create df['Min_C'] using df['Min'] (remember the parentheses!)


# 2. Preview the table to check your work

---
## Part 4: Building an Interactive Model (Something Excel Can't Easily Do)

Geoscientists constantly compare **real-world measurements** against **mathematical models**. Because seasons repeat every 365.25 days, we can approximate Sacramento's annual temperature cycle with a simple cosine wave:

$$T(t) = T_{\text{avg}} - A \cdot \cos\left(\frac{2\pi (t - \phi)}{365.25}\right)$$

where:
* $T_{\text{avg}} = (T_{\max} + T_{\min}) / 2$ is the yearly average temperature
* $A = (T_{\max} - T_{\min}) / 2$ is the seasonal amplitude (half the swing from winter to summer)
* $\phi$ (`phase_shift_days`) shifts the peak later into July/August to account for **seasonal lag** (just like how the hottest time of day is mid-afternoon, not noon!)

### Try the sliders below!
In Google Colab, the `#@param` comments turn variables into **interactive sliders**. Drag `Tmax`, `Tmin`, and `phase_shift_days`, then re-run the cell (`Shift + Enter`) to visually fit the black model curve to either the **Max** or **Min** Sacramento data!

In [ ]:
#@title Interactive Seasonal Temperature Model { run: "auto" }
Tmax = 96 #@param {type:"slider", min:40, max:120, step:1}
Tmin = 30 #@param {type:"slider", min:20, max:80, step:1}
phase_shift_days = 30 #@param {type:"slider", min:-60, max:60, step:5}

# Use the actual day-of-year (1 to 366) from each Date so the model aligns perfectly with the data
day_of_year = df['Date'].dt.dayofyear

# Calculate the cosine temperature model using np.pi (much more accurate than typing 3.14!)
T_avg = (Tmax + Tmin) / 2
amplitude = (Tmax - Tmin) / 2
temp_model = T_avg - amplitude * np.cos(2 * np.pi * (day_of_year - phase_shift_days) / 365.25)

# Plot the real Sacramento data AND our model on the exact same Date axis
plt.figure(figsize=(9, 4), dpi=150)
plt.plot(df['Date'], df['Max'], 'r^', markerfacecolor='none', alpha=0.45, label='Observed Max')
plt.plot(df['Date'], df['Min'], 'bv', markerfacecolor='none', alpha=0.45, label='Observed Min')
plt.plot(df['Date'], temp_model, 'k-', linewidth=2.5, label=f'Model ({Tmin}°F – {Tmax}°F)')

plt.title('Observed Sacramento Temperatures vs. Seasonal Cosine Model')
plt.xlabel('Year')
plt.ylabel('Temperature (°F)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

> **Quick check:** Notice how we plotted `temp_model` against `df['Date']` instead of a plain list of numbers `0, 1, 2...`—that's why the model lines up cleanly with the calendar years on the x-axis!
>
> **Mini-challenge:** Adjust the sliders above until the black curve fits the **red (Max)** points as closely as you can. Then try fitting the **blue (Min)** points. What `phase_shift_days` works best?

---
## Part 5: Multi-Tab Excel Files & Linear Regression

Most geoscience datasets you'll encounter in classes and jobs live in multi-tab Excel workbooks (`.xlsx`). Python can inspect all the tab names and load any sheet directly—no downloading required.

In [ ]:
# Load an Excel workbook from GitHub and list all of its sheet (tab) names
url_excel = 'https://github.com/sms0832/Geol11_F24/raw/refs/heads/main/DataSets1.xlsx'
print("Available tabs in DataSets1.xlsx:", pd.ExcelFile(url_excel).sheet_names)

# Load the first tab (sheet_name=0, or you can use the tab's name in quotes)
df2 = pd.read_excel(url_excel, sheet_name=0)
df2.head()

In [ ]:
# Always check exact column names (spaces, capitalization, and units matter!)
print("Columns:", list(df2.columns))

# Plot Year vs. Mass (GT) with clean labels
plt.figure(figsize=(8, 4), dpi=150)
plt.plot(df2['Year'], df2['Mass (GT)'], 'o-', color='tab:blue', linewidth=1.8, markersize=4)
plt.title('Ice Mass Change Over Time ')
plt.xlabel('Year')
plt.ylabel('Mass (GT)')
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

### Step 5.1: Combining two spreadsheets & fitting a linear regression
Suppose we want to compare **Antarctic Ice Mass** (Tab 0 of `DataSets2.xlsx`) against **Atmospheric $\text{CO}_2$** (Tab 3 of `DataSets2.xlsx`).

In Excel, aligning two datasets that cover different year ranges is tedious and error-prone. In Python, `pd.merge(..., on='Year')` acts like an automatic, bulletproof **`VLOOKUP`**—it keeps only the years present in *both* tables and lines them up row-for-row!

In [ ]:
# Inspect the tabs in DataSets2.xlsx
url_climate = 'https://github.com/sms0832/Geol11_F24/raw/refs/heads/main/DataSets2.xlsx'
print("Available tabs in DataSets2.xlsx:", pd.ExcelFile(url_climate).sheet_names)

# Load Antarctic Ice Mass (sheet 0) and Atmospheric CO2 (sheet 3)
Ant_Ice = pd.read_excel(url_climate, sheet_name=0)
CO2 = pd.read_excel(url_climate, sheet_name=3)

print(f"Antarctic Ice years: {Ant_Ice['Year'].min()} to {Ant_Ice['Year'].max()}")
print(f"CO2 years:           {CO2['Year'].min()} to {CO2['Year'].max()}")

# Merge both tables on 'Year' so every row matches the exact same year
climate = pd.merge(Ant_Ice, CO2, on='Year')
climate.head()

### Step 5.2: Fit a linear trendline and display the equation + $R^2$
Using `linregress(x, y)` gives us the **slope**, **intercept**, **$R^2$**, and **p-value** in a single line—just like Excel's *"Display Equation and $R^2$ on chart"*, except we also get the exact numbers printed cleanly for our lab report.

In [ ]:
# Pick our X and Y variables (drop any rows with missing values just in case)
clean = climate[['Average_CO2_ppm', 'MeanMass(GT)']].dropna()
x = clean['Average_CO2_ppm']
y = clean['MeanMass(GT)']

# Fit a straight line: y = m*x + b
res = linregress(x, y)
m = res.slope
b = res.intercept
r_squared = res.rvalue ** 2

print(f"Equation:  y = {m:.2f}x + ({b:.2f})")
print(f"R-squared: {r_squared:.3f}")
print(f"p-value:   {res.pvalue:.2e}")

# Plot the scatter points and dashed regression line
plt.figure(figsize=(8, 5), dpi=150)
plt.scatter(x, y, color='tab:blue', edgecolor='white', s=65, alpha=0.85, label='Annual Observations')
plt.plot(x, m * x + b, '--k', linewidth=2, label=f'Linear Fit: y = {m:.1f}x + {b:.0f}  ($R^2$ = {r_squared:.2f})')

plt.title('Antarctic Ice Mass vs. Atmospheric CO₂')
plt.xlabel('Average Atmospheric CO₂ (ppm)')
plt.ylabel('Mean Antarctic Ice Mass Anomaly (GT)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

---
## Part 6: Bookmark This! Your Reusable Geoscience Plotting Template

Save a copy of this notebook to your Google Drive (**File → Save a copy in Drive**). Whenever you need a crisp, labeled plot for a future geology lab report, poster, or senior thesis, **copy the single cell below** and only edit the **`USER SETTINGS`** section at the top!

It automatically handles:
* Scatter plots, line plots, or both
* Optional linear trendline with equation and $R^2$ in the legend
* High-resolution (300 DPI) PNG export ready to paste into Word, Google Docs, or PowerPoint

In [ ]:
# ==============================================================================
# REUSABLE GEOSCIENCE PLOTTING TEMPLATE (Edit ONLY the "USER SETTINGS" block!)
# ==============================================================================

# --- 1. LOAD YOUR DATA (uncomment the option you need) ---
data = climate                                          # Option A: Use a DataFrame already in memory
# data = pd.read_csv("paste_your_csv_url_or_file.csv")  # Option B: Load a CSV file
# data = pd.read_excel("your_file.xlsx", sheet_name=0)  # Option C: Load an Excel tab

# --- 2. USER SETTINGS: Pick your columns & labels ---
x_col = 'Average_CO2_ppm'                               # Exact column name for X-axis
y_col = 'MeanMass(GT)'                                  # Exact column name for Y-axis

plot_title  = 'Antarctic Ice Mass vs. Atmospheric CO₂'  # Figure title
x_label     = 'Average Atmospheric CO₂ (ppm)'           # X-axis label (with units!)
y_label     = 'Mean Antarctic Ice Mass Anomaly (GT)'    # Y-axis label (with units!)

plot_type     = 'scatter'   # Choose: 'scatter', 'line', or 'both'
color         = 'tab:blue'  # Try: 'tab:blue', 'tab:red', 'tab:green', 'darkorange', 'purple'
add_trendline = True        # Set to True to add a linear fit + R², or False to turn it off
save_figure   = False       # Set to True to download a 300-DPI PNG for your lab report
output_name   = 'my_geology_plot.png'

# ==============================================================================
# PLOTTING ENGINE (No need to edit below this line unless you want to customize!)
# ==============================================================================
plot_df = data[[x_col, y_col]].dropna()
x_vals = plot_df[x_col]
y_vals = plot_df[y_col]

plt.figure(figsize=(8, 5), dpi=150)

if plot_type == 'scatter':
    plt.scatter(x_vals, y_vals, color=color, edgecolor='white', s=65, alpha=0.85, label='Data')
elif plot_type == 'line':
    plt.plot(x_vals, y_vals, '-', color=color, linewidth=2, label='Data')
elif plot_type == 'both':
    plt.plot(x_vals, y_vals, 'o-', color=color, linewidth=1.8, markersize=5, label='Data')

if add_trendline:
    fit = linregress(x_vals, y_vals)
    r2 = fit.rvalue ** 2
    plt.plot(
        x_vals,
        fit.slope * x_vals + fit.intercept,
        '--k',
        linewidth=2,
        label=f'Fit: y = {fit.slope:.2f}x + {fit.intercept:.1f} ($R^2$ = {r2:.2f})',
    )

plt.title(plot_title, fontsize=13, fontweight='bold')
plt.xlabel(x_label, fontsize=11)
plt.ylabel(y_label, fontsize=11)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()

if save_figure:
    plt.savefig(output_name, dpi=300, bbox_inches='tight')
    print(f"Saved high-resolution figure as '{output_name}' (check the Files folder on the left!)")

plt.show()

### Your Turn #2 (Capstone): Plot a New Dataset Using the Template!
Put your new template to work on a different tab from `DataSets1.xlsx` or `DataSets2.xlsx`:
1. Look back at the printed tab names from `DataSets1.xlsx` or `DataSets2.xlsx` and pick one we haven't plotted yet.
2. Load that tab with `pd.read_excel(...)` and check its column names with `.columns`.
3. Copy the **Reusable Geoscience Plotting Template** from above, paste it below, update the `USER SETTINGS` block, and run it!

In [ ]:
# Step 1: Load a different tab from url_excel or url_climate
# Example: my_df = pd.read_excel(url_climate, sheet_name=1)


# Step 2: Check its column names so you spell them exactly right
# print(list(my_df.columns))


# Step 3: Copy & paste the template cell above, set data = my_df, and customize!